# Bundled Exasol lesson: Basic Udf With Gpu

Requires a GPU-capable Exasol host and GPU SLC. The local Mac kit does not provision these.

Run cells from top to bottom. In Marimo each editable cell has its own Run button; opening a lesson executes no example code. Widget steps retain their Python code; use the matching Jupyter notebook for the original widget UI.

For dashboards use the kit Data Scientist 03_share flow on Dash Server.

Source: [Exasol Notebook Connector](https://github.com/exasol/notebook-connector/blob/14e2c7ad088b7109aa21d283e98f3126bbbe27e5/exasol/nb_connector/resources/notebooks/gpu_in_udf/basic_udf_with_gpu.ipynb). MIT license: LICENSE.txt.

In [ ]:
from exakit.nb.example_runtime import bootstrap
ai_lab_config = bootstrap(globals(), 'gpu_in_udf/basic_udf_with_gpu.ipynb')

<div style="text-align: right;">
  <img src="https://raw.githubusercontent.com/exasol/ai-lab/refs/heads/main/assets/Exasol_Logo_2025_Dark.svg" style="width:200px; margin: 10px;" />
</div>

## Basic UDF using GPU Acceleration

In this tutorial we will build two very simple UDFs which demonstrate that GPU detection works.

## Open Secure Configuration Storage


In [ ]:
print("Using the Starter Kit encrypted connection; tutorial schema: EXAKIT_EXAMPLES.")

### Instantiate ScriptLanguagesContainer

The following cell creates an instance of class `ScriptLanguageContainer` from the notebook-connector,
which enables us to use the Script Language Container (SLC).

In [ ]:
from exasol.nb_connector.slc import ScriptLanguageContainer
slc = ScriptLanguageContainer(secrets=ai_lab_config, name="gpu_slc")

### Connect to the Database and activate the Container

In [ ]:
from exasol.nb_connector.language_container_activation import open_pyexasol_connection_with_lang_definitions

conn = open_pyexasol_connection_with_lang_definitions(ai_lab_config, schema=ai_lab_config.db_schema, compression=True)

### Create GPU enabled UDFs

Exasol UDFs provide the script option `%perInstanceRequiredAcceleratorDevices` which allows us to define the behavior of the database when executing UDF's. This option can have the following values:
* `None`: NO accelerator selected/required (implicit default for CPU processing)
* `GpuNvidia`: NVIDIA GPU accelerator required by the UDF. The invocation of the UDF will fail if the GPU is unavailable or the Script Language Container is not suited for GPU's.
* `None|GpuNvidia`: NVIDIA GPU accelerator preferred but fallback to NO accelerator (CPU processing)

The following sample UDF requires GPU support by adding the line:
```
 %perInstanceRequiredAcceleratorDevices GpuNvidia;
```
and emits the value of the environment variable `NVIDIA_VISIBLE_DEVICES`.

In [ ]:
import textwrap
conn.execute(textwrap.dedent(f"""
CREATE OR REPLACE {slc.language_alias} SCALAR SCRIPT
nvidia_env_variable_check()
EMITS (var_name VARCHAR(1000), var_value VARCHAR(1000)) AS
 %perInstanceRequiredAcceleratorDevices GpuNvidia;

import os

def run(ctx):
    ctx.emit("NVIDIA_VISIBLE_DEVICES", os.getenv('NVIDIA_VISIBLE_DEVICES', "<invalid>"))
/
"""))

res = conn.export_to_pandas("SELECT nvidia_env_variable_check();")
res

The next UDF leverages the `pytorch` package which was installed in our custom Script Language Container and returns the string "GPU Found" if the GPU was detected in the UDF by `pytorch`.

In [ ]:
import textwrap
conn.execute(textwrap.dedent(f"""
CREATE OR REPLACE {slc.language_alias} SCALAR SCRIPT
gpu_available()
RETURNS VARCHAR(10000) AS
 %perInstanceRequiredAcceleratorDevices GpuNvidia;

import torch

def run(ctx):
    if torch.cuda.is_available():
        return "GPU Found"
    else:
        return "GPU Not Found"
/
"""))

res = conn.export_to_pandas("SELECT gpu_available();")
res

Now, try out the other values for `%perInstanceRequiredAcceleratorDevices` and see what happens.

### Finish
Now you can continue with [Advanced UDF with GPU support](./advanced_udf_with_gpu.ipynb)